# 04 — Backtest Model Portfolio

This notebook converts the final trend signal and selected volatility forecasts into monthly portfolio returns.

The backtest is evaluated out-of-sample from 2018 onward. Portfolio weights are formed using information available at each month-end and are applied to the following month's returns.

In [12]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.optimize import minimize
except ImportError:
    minimize = None

pd.options.display.float_format = "{:,.4f}".format

DATA_DIR = Path("data")
RESULTS_DIR = Path("results")

TREND_DIR = RESULTS_DIR / "trend_signal"
VOL_DIR = RESULTS_DIR / "volatility_forecast"
OUT_DIR = RESULTS_DIR / "backtest_model_portfolio"

OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Inputs

The backtest uses monthly asset returns, benchmark returns, the final trend signal, and the selected monthly volatility forecasts from the previous notebooks.

In [17]:
def find_date_column(df):
    candidates = ["date", "dates", "datetime", "month", "rebalance_date"]
    lower_map = {str(c).lower(): c for c in df.columns}

    for c in candidates:
        if c in lower_map:
            return lower_map[c]

    return df.columns[0]


def read_time_series(path, value_names=("value", "return", "signal", "vol", "forecast")):
    df = pd.read_csv(path)
    date_col = find_date_column(df)

    df[date_col] = pd.to_datetime(df[date_col])

    lower_map = {str(c).lower(): c for c in df.columns}
    asset_col = next((lower_map[c] for c in ["asset", "ticker", "symbol"] if c in lower_map), None)
    value_col = next((lower_map[c] for c in value_names if c in lower_map), None)

    if asset_col is not None and value_col is not None:
        df = df.pivot_table(
            index=date_col,
            columns=asset_col,
            values=value_col,
            aggfunc="last"
        )
    else:
        df = df.set_index(date_col)

    df = df.sort_index()
    df = df[~df.index.duplicated(keep="last")]
    df = df.apply(pd.to_numeric, errors="coerce")

    return df


def to_decimal_returns(df, label="returns"):
    values = df.stack().dropna().abs()

    if not values.empty and values.quantile(0.95) > 1.0:
        print(f"Converted {label} from percent units to decimal units.")
        return df / 100.0

    return df


def to_decimal_vol(df, label="volatility forecasts"):
    values = df.stack().dropna().abs()

    if not values.empty and values.median() > 2.0:
        print(f"Converted {label} from percent units to decimal units.")
        return df / 100.0

    return df


def to_month_end_last(df):
    out = df.copy()
    out.index = pd.to_datetime(out.index).to_period("ME").to_timestamp("ME")
    return out.groupby(level=0).last().sort_index()


def to_monthly_returns(df):
    df = df.sort_index()
    gaps = df.index.to_series().diff().dropna()

    if not gaps.empty and gaps.median() < pd.Timedelta(days=20):
        return ((1.0 + df).resample("M").prod() - 1.0).sort_index()

    return to_month_end_last(df)

In [19]:
monthly_returns = read_time_series(TREND_DIR / "monthly_returns_aligned.csv")
benchmark_returns = read_time_series(DATA_DIR / "benchmark_returns.csv")
final_signal = read_time_series(TREND_DIR / "final_signal.csv")
vol_forecast = read_time_series(VOL_DIR / "final_selected_vol_forecast_monthly_oos.csv")

#monthly_returns = to_monthly_returns(to_decimal_returns(monthly_returns, "monthly returns"))
#benchmark_returns = to_monthly_returns(to_decimal_returns(benchmark_returns, "benchmark returns"))
#final_signal = to_month_end_last(final_signal)
#vol_forecast = to_month_end_last(to_decimal_vol(vol_forecast, "volatility forecasts"))

data_audit = pd.DataFrame({
    "Start": [
        monthly_returns.index.min(),
        benchmark_returns.index.min(),
        final_signal.index.min(),
        vol_forecast.index.min(),
    ],
    "End": [
        monthly_returns.index.max(),
        benchmark_returns.index.max(),
        final_signal.index.max(),
        vol_forecast.index.max(),
    ],
    "Rows": [
        len(monthly_returns),
        len(benchmark_returns),
        len(final_signal),
        len(vol_forecast),
    ],
    "Columns": [
        monthly_returns.shape[1],
        benchmark_returns.shape[1],
        final_signal.shape[1],
        vol_forecast.shape[1],
    ],
}, index=[
    "Monthly asset returns",
    "Benchmark returns",
    "Final trend signal",
    "Selected volatility forecasts",
])

display(data_audit)

,Start,End,Rows,Columns
Monthly asset returns,2007-05-31,2026-06-30,230,11
Benchmark returns,2007-04-12,2026-06-05,4819,2
Final trend signal,2007-05-31,2026-06-30,230,11
Selected volatility forecasts,2018-01-31,2026-06-30,102,11


## 2. Backtest Assumptions

The strategy is rebalanced monthly. Signals and volatility forecasts are treated as month-end information and are applied with a one-month lag. No model selection or parameter tuning is performed during the OOS period.

In [20]:
OOS_START = pd.Timestamp("2018-01-31")

CASH_ASSET = "SHY"

WEIGHT_LAG_MONTHS = 1

RP_WINDOW_MONTHS = 36
RP_MIN_OBS = 24

CORR_WINDOW_MONTHS = 36
CORR_MIN_OBS = 24

TARGET_VOL = 0.10
MAX_GROSS_EXPOSURE = 1.00
ALLOW_VOL_TARGET_UPSCALING = False

TRANSACTION_COST_BPS = 5.0

## 3. Align Data and Enforce Timing

The portfolio universe is restricted to assets available in the return, signal, and volatility forecast files. SHY is treated as the defensive cash proxy when available.

In [21]:
common_assets = [
    c for c in monthly_returns.columns
    if c in final_signal.columns and c in vol_forecast.columns
]

if len(common_assets) == 0:
    raise ValueError("No common assets found across returns, signal, and volatility forecast files.")

cash_asset = CASH_ASSET if CASH_ASSET in common_assets else None
risk_assets = [a for a in common_assets if a != cash_asset]

if len(risk_assets) == 0:
    raise ValueError("No risk assets available after excluding the cash proxy.")

monthly_returns = monthly_returns[common_assets].sort_index()
final_signal = final_signal.reindex(columns=common_assets).sort_index()
vol_forecast = vol_forecast.reindex(columns=common_assets).sort_index()

asof_signal = final_signal.reindex(monthly_returns.index).ffill()
asof_vol = vol_forecast.reindex(monthly_returns.index).ffill()

alignment_audit = pd.DataFrame({
    "Included": [len(common_assets), len(risk_assets), 1 if cash_asset is not None else 0],
}, index=["Total assets", "Risk assets", "Cash proxy"])

display(alignment_audit)

print("Risk assets:", risk_assets)
print("Cash proxy:", cash_asset)
print("OOS start:", OOS_START.date())

,Included
Total assets,11
Risk assets,10
Cash proxy,1


Risk assets: ['DBC', 'EEM', 'EFA', 'GLD', 'HYG', 'IEF', 'LQD', 'SPY', 'TLT', 'VNQ']
Cash proxy: SHY
OOS start: 2018-01-31


In [22]:
signal_check = asof_signal.loc[OOS_START:, common_assets].describe().T[["min", "mean", "max"]]
vol_check = asof_vol.loc[OOS_START:, common_assets].describe().T[["min", "mean", "max"]]

print("Signal summary:")
display(signal_check)

print("Volatility forecast summary:")
display(vol_check)

Signal summary:


,min,mean,max
DBC,0.0000,0.6013,1.0000
EEM,0.0000,0.6078,1.0000
EFA,0.0000,0.6732,1.0000
GLD,0.0000,0.7288,1.0000
HYG,0.0000,0.7745,1.0000
IEF,0.0000,0.5327,1.0000
LQD,0.0000,0.6405,1.0000
SHY,0.0000,0.7092,1.0000
SPY,0.0000,0.7974,1.0000
TLT,0.0000,0.4575,1.0000


Volatility forecast summary:


,min,mean,max
DBC,0.0977,0.1766,0.3748
EEM,0.1390,0.2093,0.7414
EFA,0.0964,0.1692,0.7093
GLD,0.1067,0.1625,0.4117
HYG,0.0414,0.0755,0.4143
IEF,0.0390,0.0650,0.1487
LQD,0.0439,0.0720,0.3807
SHY,0.0025,0.0139,0.0502
SPY,0.0937,0.1771,0.7701
TLT,0.0958,0.1438,0.4305


## 4. Risk Parity Benchmark

The Risk Parity benchmark uses rolling monthly covariance estimates based only on returns available at each rebalance date. SHY is excluded from the Risk Parity asset set when it is used as the cash proxy.

In [23]:
def inverse_vol_fallback(cov):
    diag = pd.Series(np.diag(cov.values), index=cov.index).clip(lower=0.0)
    inv_vol = 1.0 / np.sqrt(diag.replace(0.0, np.nan))
    inv_vol = inv_vol.replace([np.inf, -np.inf], np.nan).dropna()

    if inv_vol.empty:
        return pd.Series(1.0 / len(cov.index), index=cov.index)

    weights = inv_vol / inv_vol.sum()
    return weights.reindex(cov.index).fillna(0.0)


def solve_erc_weights(cov):
    cov = cov.astype(float).copy()
    cov = cov.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    cov += np.eye(len(cov)) * 1e-10

    fallback = inverse_vol_fallback(cov)

    if minimize is None or len(cov) == 1:
        return fallback

    cov_values = cov.values
    n = len(cov)

    def objective(w):
        port_var = float(w @ cov_values @ w)

        if port_var <= 0 or not np.isfinite(port_var):
            return 1e6

        risk_contrib = w * (cov_values @ w) / port_var
        return float(((risk_contrib - 1.0 / n) ** 2).sum())

    res = minimize(
        objective,
        x0=fallback.values,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n,
        constraints={"type": "eq", "fun": lambda w: np.sum(w) - 1.0},
        options={"maxiter": 500, "ftol": 1e-12},
    )

    if not res.success or not np.all(np.isfinite(res.x)):
        return fallback

    weights = pd.Series(np.clip(res.x, 0.0, None), index=cov.index)
    return weights / weights.sum()


def build_rolling_risk_parity_weights(returns, risk_assets, window=36, min_obs=24):
    weights = pd.DataFrame(np.nan, index=returns.index, columns=risk_assets)

    for date in returns.index:
        hist = returns.loc[:date, risk_assets].tail(window).dropna(how="any")

        if len(hist) < min_obs:
            continue

        weights.loc[date, risk_assets] = solve_erc_weights(hist.cov())

    return weights

In [26]:
rp_risk_weights = build_rolling_risk_parity_weights(
    monthly_returns,
    risk_assets=risk_assets,
    window=RP_WINDOW_MONTHS,
    min_obs=RP_MIN_OBS,
)

rp_raw_weights = pd.DataFrame(0.0, index=rp_risk_weights.index, columns=common_assets)
rp_raw_weights.loc[:, risk_assets] = rp_risk_weights

rp_raw_weights.tail()

,DBC,EEM,EFA,GLD,HYG,IEF,LQD,SHY,SPY,TLT,VNQ
Date,,,,,,,,,,,
2026-02-28,0.1703,0.0629,0.0686,0.0974,0.1754,0.1308,0.0982,0.0000,0.0873,0.0577,0.0514
2026-03-31,0.1964,0.0573,0.0640,0.0879,0.1731,0.1338,0.0992,0.0000,0.0825,0.0580,0.0478
2026-04-30,0.1844,0.0533,0.0630,0.0964,0.1772,0.1391,0.1038,0.0000,0.0755,0.0609,0.0463
2026-05-31,0.1971,0.0525,0.0638,0.0948,0.1782,0.1363,0.1016,0.0000,0.0722,0.0588,0.0447
2026-06-30,0.1969,0.0515,0.0647,0.0873,0.1790,0.1362,0.1036,0.0000,0.0722,0.0612,0.0475


## 5. Trend + Inverse-Volatility Portfolio

Positive trend signals define the risk sleeve. Within the active sleeve, inverse forecast volatility tilts exposure toward lower-risk assets. Unused risk budget is allocated to SHY when available.

In [27]:
def prepare_signal_scores(signal_df, assets):
    scores = signal_df.reindex(columns=assets)
    scores = scores.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    scores = scores.clip(lower=0.0)

    if scores.stack().dropna().max() > 1.0:
        row_max = scores.max(axis=1).replace(0.0, np.nan)
        scores = scores.div(row_max, axis=0)

    return scores.clip(upper=1.0).fillna(0.0)


def build_trend_invvol_weights(signal, vol, all_assets, risk_assets, cash_asset=None):
    idx = signal.index.intersection(vol.index).sort_values()

    scores = prepare_signal_scores(signal.loc[idx], risk_assets)

    vol_block = vol.reindex(index=idx, columns=risk_assets)
    vol_block = vol_block.replace([np.inf, -np.inf], np.nan)

    inv_vol = 1.0 / vol_block.where(vol_block > 0.0)
    raw = scores * inv_vol

    sleeve_weights = raw.div(raw.sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
    risk_budget = scores.mean(axis=1).clip(0.0, 1.0).fillna(0.0)

    weights = pd.DataFrame(0.0, index=idx, columns=all_assets)
    weights.loc[:, risk_assets] = sleeve_weights.multiply(risk_budget, axis=0)

    if cash_asset is not None:
        weights.loc[:, cash_asset] = 1.0 - weights[risk_assets].sum(axis=1)

    return weights.clip(lower=0.0)

In [28]:
trend_invvol_raw_weights = build_trend_invvol_weights(
    signal=asof_signal,
    vol=asof_vol,
    all_assets=common_assets,
    risk_assets=risk_assets,
    cash_asset=cash_asset,
)

trend_invvol_raw_weights.loc[OOS_START:].head()

,DBC,EEM,EFA,GLD,HYG,IEF,LQD,SHY,SPY,TLT,VNQ
Date,,,,,,,,,,,
2018-01-31,0.0873,0.0517,0.0781,0.0800,0.1143,0.1406,0.1753,0.0667,0.0722,0.0813,0.0524
2018-02-28,0.0929,0.0422,0.0597,0.0980,0.1096,0.0803,0.1423,0.2333,0.0559,0.0685,0.0173
2018-03-31,0.1100,0.0490,0.0811,0.0707,0.0702,0.0000,0.0884,0.4333,0.0554,0.0419,0.0000
2018-04-30,0.0894,0.0585,0.0678,0.0916,0.0720,0.0000,0.0736,0.4667,0.0464,0.0341,0.0000
2018-05-31,0.0988,0.0421,0.0502,0.0740,0.0774,0.0000,0.0000,0.6000,0.0576,0.0000,0.0000


## 6. Volatility Targeting Overlay

The volatility overlay estimates portfolio risk using selected asset volatility forecasts and a rolling realized correlation matrix. The default setting only scales exposure down; it does not add leverage.

In [ ]:
def estimate_portfolio_vol(weights, vol, returns, assets, window=36, min_obs=24):
    est_vol = pd.Series(np.nan, index=weights.index, name="Estimated_Portfolio_Vol")

    for date in weights.index:
        w = weights.loc[date, assets].fillna(0.0).astype(float)

        v = vol.reindex(index=[date], columns=assets).iloc[0]
        v = v.replace([np.inf, -np.inf], np.nan).astype(float)
        v = v.where(v > 0.0)

        if v.isna().all() or np.isclose(w.abs().sum(), 0.0):
            continue

        v = v.fillna(v.median())

        hist = returns.loc[:date, assets].tail(window).dropna(how="any")

        if len(hist) >= min_obs:
            corr_df = hist.corr().reindex(index=assets, columns=assets).fillna(0.0)
            corr = corr_df.to_numpy().copy()
            np.fill_diagonal(corr, 1.0)
        else:
            corr = np.eye(len(assets))

        cov_ann = np.outer(v.values, v.values) * corr
        port_var = float(w.values @ cov_ann @ w.values)

        est_vol.loc[date] = np.sqrt(max(port_var, 0.0))

    return est_vol


In [31]:
trend_invvol_est_vol = estimate_portfolio_vol(
    weights=trend_invvol_raw_weights,
    vol=asof_vol,
    returns=monthly_returns,
    assets=common_assets,
    window=CORR_WINDOW_MONTHS,
    min_obs=CORR_MIN_OBS,
)

trend_invvol_vt_raw_weights, vol_target_scale = apply_vol_target(
    base_weights=trend_invvol_raw_weights,
    est_vol=trend_invvol_est_vol,
    target_vol=TARGET_VOL,
    cash_asset=cash_asset,
    max_gross_exposure=MAX_GROSS_EXPOSURE,
    allow_upscaling=ALLOW_VOL_TARGET_UPSCALING,
)

vol_target_diagnostics = pd.concat(
    [trend_invvol_est_vol, vol_target_scale],
    axis=1
)

display(vol_target_diagnostics.loc[OOS_START:].describe().T)

ValueError: underlying array is read-only